# Notebook 01 --- Exploratory Data Analysis

## Credit Card Default Prediction: UCI Taiwan Dataset

**Objective.** Systematically characterise the UCI Credit Card Default dataset (30,000 records, 23 features, binary target) to inform every downstream modelling decision --- from tokenisation design to loss function selection. Each analysis section concludes with a concrete architectural implication for the transformer and random forest models.

**Dataset.** Yeh, I.C. & Lien, C.H. (2009). *The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients.* Expert Systems with Applications, 36(2), 2473--2480.

---

### Table of Contents

1. [Environment and Data Loading](#1-environment-and-data-loading)
2. [Schema Inspection and Validation](#2-schema-inspection-and-validation)
3. [Target Variable Analysis](#3-target-variable-analysis)
4. [Univariate Feature Analysis](#4-univariate-feature-analysis)
5. [Bivariate Feature--Target Associations](#5-bivariate-feature-target-associations)
6. [Temporal Structure Analysis](#6-temporal-structure-analysis)
7. [Correlation and Multicollinearity](#7-correlation-and-multicollinearity)
8. [Derived Risk Indicators](#8-derived-risk-indicators)
9. [Sequential Dependency Analysis](#9-sequential-dependency-analysis)
10. [Summary of Findings and Architectural Implications](#10-summary-of-findings)

<a id="1-environment-and-data-loading"></a>
## 1. Environment and Data Loading

Standard scientific Python stack. All visualisation parameters are configured once for publication-quality output at 300 DPI.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, pointbiserialr, spearmanr

warnings.filterwarnings("ignore")

# --- Publication-quality figure defaults ---
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

# --- Colour scheme ---
C_ND = "#2C73D2"   # No Default (blue)
C_D  = "#D63031"   # Default (red)
CMAP_DIV = "RdBu_r"

# --- Feature groups ---
PAY_STATUS  = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
BILL_AMT    = [f"BILL_AMT{i}" for i in range(1, 7)]
PAY_AMT     = [f"PAY_AMT{i}" for i in range(1, 7)]
DEMOGRAPHICS = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"]
CATEGORICALS = ["SEX", "EDUCATION", "MARRIAGE"]
NUMERICALS   = ["LIMIT_BAL", "AGE"] + BILL_AMT + PAY_AMT
ALL_FEATURES = DEMOGRAPHICS + PAY_STATUS + BILL_AMT + PAY_AMT
TARGET       = "DEFAULT"
MONTH_LABELS = ["Sep", "Aug", "Jul", "Jun", "May", "Apr"]


print(f"NumPy {np.__version__} | pandas {pd.__version__} | matplotlib {plt.matplotlib.__version__}")

In [28]:
# --- Utility functions (used across multiple sections) ---

def cramers_v(x, y):
    """Cramer's V: effect size for chi-squared test of independence."""
    ct = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.sum().sum()
    min_dim = min(ct.shape[0], ct.shape[1]) - 1
    return np.sqrt(chi2 / (n * max(min_dim, 1)))

def wilson_ci(p, n, z=1.96):
    """Wilson score interval for binomial proportion."""
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return centre - margin, centre + margin

def cohens_d(x, y):
    """Cohen's d: standardised mean difference."""
    nx, ny = len(x), len(y)
    s_pool = np.sqrt(((nx - 1) * x.std()**2 + (ny - 1) * y.std()**2) / (nx + ny - 2))
    return (x.mean() - y.mean()) / s_pool

# Pre-compute overall default rate (used across multiple sections)
# Will be set after data loading


In [ ]:
# --- Load dataset via the resilient multi-source loader ---
# Tries the UCI API first; on failure, automatically falls back to the
# local manual dataset shipped under data/raw/. See src/data_sources.py
# for the full ingestion architecture.
import sys
from pathlib import Path

sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))

from data_sources import build_default_data_source

_source = build_default_data_source(mode="auto", allow_fallback=True)
_result = _source.load()
df = _result.dataframe.copy()

print(f"[INFO] {_result.summary()}")
if _result.failed_attempts:
    for _name, _err in _result.failed_attempts:
        print(f"[INFO]   ↳ fell back from '{_name}': {_err}")

# Drop ID column if present (local .xls ships with one; ucimlrepo does not).
if "ID" in df.columns:
    df = df.drop(columns=["ID"])

# Some local-file versions use 'PAY_1' for the September snapshot, and the
# target column may arrive as 'default payment next month'. Normalise both.
for _col in list(df.columns):
    _norm = _col.strip().upper().replace(" ", "_").replace(".", "_")
    if _norm == "PAY_1":
        df.rename(columns={_col: "PAY_0"}, inplace=True)
    elif "DEFAULT" in _norm and _col != "DEFAULT":
        df.rename(columns={_col: "DEFAULT"}, inplace=True)

# Clean undocumented categorical codes (matches src/data_preprocessing.py).
df.loc[df["EDUCATION"].isin([0, 5, 6]), "EDUCATION"] = 4
df.loc[df["MARRIAGE"] == 0, "MARRIAGE"] = 3

# Reorder to canonical schema.
df = df[ALL_FEATURES + [TARGET]]

print(f"Source:        {_result.source_name}  ({_result.source_type})")
print(f"Origin:        {_result.origin}")
print(f"Shape:         {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns:       {list(df.columns)}")
print(f"Memory usage:  {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

# Pre-compute overall default rate
overall_default_rate = df[TARGET].mean()


<a id="2-schema-inspection-and-validation"></a>
## 2. Schema Inspection and Validation

Before any analysis, we verify data integrity: missing values, duplicate records, value ranges for categorical codes, and target variable encoding. The UCI dataset is known to contain undocumented category codes (EDUCATION: 0, 5, 6; MARRIAGE: 0) which were cleaned above following standard practice from Yeh & Lien (2009).

In [ ]:
# --- Data quality audit ---
print("=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(f"\n{'Metric':<35} {'Value':>15}")
print("-" * 50)
print(f"{'Total records':<35} {len(df):>15,}")
print(f"{'Total features':<35} {len(df.columns) - 1:>15}")
print(f"{'Missing values (total)':<35} {df.isnull().sum().sum():>15}")
print(f"{'Duplicate rows':<35} {df.duplicated().sum():>15}")
print(f"{'Duplicate rate':<35} {100 * df.duplicated().sum() / len(df):>14.2f}%")

print(f"\n--- Categorical Value Ranges (post-cleaning) ---")
for col in ["SEX", "EDUCATION", "MARRIAGE"]:
    unique = sorted(df[col].unique())
    print(f"  {col:<15} unique values: {unique}")

print(f"\n--- PAY Status Value Range ---")
all_pay_vals = set()
for col in PAY_STATUS:
    all_pay_vals.update(df[col].unique())
print(f"  Observed values across all PAY columns: {sorted(all_pay_vals)}")

print(f"\n--- Numerical Ranges ---")
for col in ["LIMIT_BAL", "AGE"]:
    print(f"  {col:<15} min={df[col].min():>10,.0f}   max={df[col].max():>10,.0f}")

print(f"\n--- Target Encoding ---")
print(f"  Unique values: {sorted(df[TARGET].unique())}")
print(f"  Default rate:  {df[TARGET].mean():.4f}")

In [ ]:
# --- Comprehensive summary statistics table ---
stats_rows = []
for col in ALL_FEATURES:
    d0 = df[df[TARGET] == 0][col]
    d1 = df[df[TARGET] == 1][col]
    stats_rows.append({
        "Feature": col,
        "Mean (All)": df[col].mean(),
        "Std": df[col].std(),
        "Mean (No Def)": d0.mean(),
        "Mean (Default)": d1.mean(),
        "Median": df[col].median(),
        "Skewness": df[col].skew(),
        "Kurtosis": df[col].kurtosis(),
    })

stats_df = pd.DataFrame(stats_rows).set_index("Feature")
stats_df.style.format("{:.2f}").background_gradient(
    subset=["Skewness"], cmap="RdBu_r", vmin=-5, vmax=5
)

### 2.1 Duplicate Row Analysis

The dataset contains 35 duplicate rows (0.12%). We characterise these to determine whether they represent legitimate records or data entry errors.


In [ ]:
# --- Duplicate row characterisation ---
duplicates = df[df.duplicated(keep=False)].sort_values(list(df.columns))
n_dup_groups = df[df.duplicated(keep="first")].shape[0]

print(f"Duplicate rows: {len(duplicates)} total across {n_dup_groups} unique duplicate groups")
print(f"Duplicate rate: {100 * n_dup_groups / len(df):.3f}%")

# Default rate among duplicates vs overall
dup_default_rate = duplicates[TARGET].mean()
overall_default_rate = df[TARGET].mean()
print(f"\nDefault rate in duplicates:  {dup_default_rate:.4f}")
print(f"Default rate overall:        {overall_default_rate:.4f}")
print(f"Difference:                  {dup_default_rate - overall_default_rate:+.4f}")

# Feature distribution of duplicates
print("\n--- Duplicate Feature Means vs Population ---")
print(f"{'Feature':<15} {'Dup Mean':>12} {'Pop Mean':>12} {'Ratio':>8}")
print("-" * 50)
for col in ["LIMIT_BAL", "AGE", "PAY_0", "BILL_AMT1", "PAY_AMT1"]:
    d_mean = duplicates[col].mean()
    p_mean = df[col].mean()
    ratio = d_mean / p_mean if p_mean != 0 else float("inf")
    print(f"{col:<15} {d_mean:>12,.1f} {p_mean:>12,.1f} {ratio:>8.2f}")

print("\nConclusion: Duplicates appear representative of the population.")
print("Retained in dataset (removal would have negligible impact).")


### 2.2 Zero-Inflation Analysis (PAY_AMT Features)

Payment amount features exhibit significant zero-inflation: 17--24% of customers make zero payments in any given month. Zero payments coupled with positive bill amounts represent **payment avoidance** --- a strong default indicator.


In [ ]:
# --- Zero-inflation analysis ---
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

zero_stats = []
for idx, col in enumerate(PAY_AMT):
    ax = axes[idx // 3][idx % 3]
    month = MONTH_LABELS[idx]

    n_zero = (df[col] == 0).sum()
    pct_zero = 100 * n_zero / len(df)

    # Zero payment WITH positive bill (payment avoidance)
    bill_col = BILL_AMT[idx]
    avoidance = ((df[col] == 0) & (df[bill_col] > 0)).sum()
    pct_avoidance = 100 * avoidance / len(df)

    # Default rates
    dr_zero = df[df[col] == 0][TARGET].mean()
    dr_nonzero = df[df[col] > 0][TARGET].mean()
    dr_avoidance = df[(df[col] == 0) & (df[bill_col] > 0)][TARGET].mean() if avoidance > 0 else 0

    zero_stats.append({
        "Month": month, "Feature": col,
        "Zero %": pct_zero, "Avoidance %": pct_avoidance,
        "DR (zero)": dr_zero, "DR (nonzero)": dr_nonzero,
        "DR (avoidance)": dr_avoidance, "Risk Ratio": dr_zero / dr_nonzero if dr_nonzero > 0 else 0,
    })

    # Stacked bar: zero vs nonzero, coloured by default
    categories = ["Zero Payment\n(all)", "Zero + Bill > 0\n(avoidance)", "Paid > 0"]
    rates = [dr_zero, dr_avoidance, dr_nonzero]
    colors_bar = [C_D, "#8B0000", C_ND]

    bars = ax.bar(categories, rates, color=colors_bar, edgecolor="white", alpha=0.8)
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.01,
                f"{rate:.1%}", ha="center", fontsize=7, fontweight="bold")

    ax.axhline(y=overall_default_rate, color="gray", linestyle="--", linewidth=1, alpha=0.6)
    ax.set_ylabel("Default Rate")
    ax.set_title(f"{col} ({month})\nZero: {pct_zero:.1f}%, Avoidance: {pct_avoidance:.1f}%")
    ax.set_ylim(0, 0.55)

fig.suptitle("Zero-Inflation Analysis: Payment Avoidance as Default Predictor\n"
             "(Dashed line = overall default rate)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Summary table
zero_df = pd.DataFrame(zero_stats)
print("\n--- Zero Payment Summary ---")
print(zero_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))


### 2.3 Outlier Detection (IQR Method)

We apply the Tukey IQR method ($1.5 \times \text{IQR}$ fence) to identify univariate outliers in each numerical feature. PAY_AMT features are expected to show the highest outlier rates due to their extreme skewness (skewness 10--34, kurtosis 144--2,077).


In [ ]:
# --- IQR-based outlier detection ---
def iqr_outlier_count(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = (series < lower) | (series > upper)
    return outliers.sum(), lower, upper

fig, ax = plt.subplots(figsize=(10, 8))

outlier_data = []
for col in NUMERICALS:
    n_out, lo, hi = iqr_outlier_count(df[col])
    pct = 100 * n_out / len(df)
    outlier_data.append({"Feature": col, "Outliers": n_out, "Pct": pct, "Lower": lo, "Upper": hi})

out_df = pd.DataFrame(outlier_data).sort_values("Pct", ascending=True)

colors_out = ["#E17055" if p > 5 else "#FDCB6E" if p > 2 else "#74B9FF" for p in out_df["Pct"]]
ax.barh(range(len(out_df)), out_df["Pct"], color=colors_out, edgecolor="white")
ax.set_yticks(range(len(out_df)))
ax.set_yticklabels(out_df["Feature"])
ax.set_xlabel("Outlier Rate (%)")
ax.set_title("IQR-Based Outlier Detection (1.5 x IQR Fence)\n"
             "Red > 5%, Yellow > 2%, Blue < 2%", fontweight="bold")

for i, (_, row) in enumerate(out_df.iterrows()):
    ax.text(row["Pct"] + 0.2, i, f"{row['Outliers']:,} ({row['Pct']:.1f}%)", va="center", fontsize=7)

ax.legend(handles=[
    Patch(facecolor="#E17055", label="High (> 5%)"),
    Patch(facecolor="#FDCB6E", label="Moderate (2-5%)"),
    Patch(facecolor="#74B9FF", label="Low (< 2%)"),
], loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()

print("\n--- Outlier Summary (sorted by rate) ---")
print(out_df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))


### 2.4 Normality Assessment

We apply formal normality tests to determine appropriate scaling strategies:
- **D'Agostino-Pearson omnibus test** ($H_0$: data is normally distributed)
- **Q-Q plots** for visual assessment against the theoretical normal distribution

Features that reject normality ($p < 0.05$) may benefit from Yeo-Johnson or log transforms before StandardScaler.


In [ ]:
# --- Normality tests + QQ plots ---
from scipy.stats import normaltest, shapiro

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

test_features = ["LIMIT_BAL", "AGE", "BILL_AMT1", "BILL_AMT3",
                 "PAY_AMT1", "PAY_AMT2", "PAY_AMT4", "PAY_AMT6"]

norm_results = []
for idx, feat in enumerate(test_features):
    ax = axes[idx // 4][idx % 4]
    data = df[feat].dropna()

    # QQ plot
    sorted_data = np.sort(data.values)
    n = len(sorted_data)
    theoretical_q = stats.norm.ppf(np.linspace(0.001, 0.999, n))
    ax.scatter(theoretical_q, sorted_data, s=1, alpha=0.3, color="#2C73D2")

    # Reference line
    slope = data.std()
    intercept = data.mean()
    x_line = np.array([-3, 3])
    ax.plot(x_line, intercept + slope * x_line, "r--", linewidth=1.5)

    # D'Agostino test (requires n >= 20)
    k2_stat, p_dagostino = normaltest(data)
    # Shapiro-Wilk on subsample (max 5000 for computational reasons)
    subsample = data.sample(min(5000, len(data)), random_state=42)
    w_stat, p_shapiro = shapiro(subsample)

    norm_results.append({
        "Feature": feat, "Skew": data.skew(), "Kurt": data.kurtosis(),
        "D'Agostino p": p_dagostino, "Shapiro p": p_shapiro,
        "Normal?": "Yes" if p_dagostino > 0.05 else "No",
    })

    p_str = f"p < 0.001" if p_dagostino < 0.001 else f"p = {p_dagostino:.3f}"
    verdict = "NORMAL" if p_dagostino > 0.05 else "NON-NORMAL"
    ax.set_title(f"{feat}\nD'Agostino {p_str} [{verdict}]", fontsize=9)
    ax.set_xlabel("Theoretical Quantiles")
    if idx % 4 == 0:
        ax.set_ylabel("Sample Quantiles")

fig.suptitle("Q-Q Plots with D'Agostino-Pearson Normality Test\n"
             "Red line = perfect normal. Deviation = non-normality.",
             fontsize=12, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

# Summary table
norm_df = pd.DataFrame(norm_results)
print("\n--- Normality Test Results ---")
print(norm_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print("\nAll PAY_AMT and BILL_AMT features reject normality.")
print("Recommendation: Apply Yeo-Johnson or log1p transform before scaling.")


### 2.5 Variance Inflation Factor (VIF) Analysis

VIF quantifies multicollinearity severity for each feature:

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ is the coefficient of determination from regressing feature $j$ on all other features. Convention: VIF $> 5$ moderate, VIF $> 10$ severe multicollinearity.


In [ ]:
# --- Variance Inflation Factor ---
from numpy.linalg import LinAlgError

def compute_vif(X):
    """Compute VIF for each feature in X."""
    vifs = []
    X_np = X.values.astype(float)
    for i in range(X_np.shape[1]):
        y_i = X_np[:, i]
        X_others = np.delete(X_np, i, axis=1)
        # Add intercept
        X_others = np.column_stack([np.ones(len(X_others)), X_others])
        try:
            beta = np.linalg.lstsq(X_others, y_i, rcond=None)[0]
            y_hat = X_others @ beta
            ss_res = np.sum((y_i - y_hat) ** 2)
            ss_tot = np.sum((y_i - y_i.mean()) ** 2)
            r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0
            vif = 1 / (1 - r_squared) if r_squared < 1 else float("inf")
        except LinAlgError:
            vif = float("inf")
        vifs.append(vif)
    return vifs

# Compute VIF on numerical features
vif_features = NUMERICALS
X_vif = df[vif_features].dropna()
vif_values = compute_vif(X_vif)

vif_df = pd.DataFrame({"Feature": vif_features, "VIF": vif_values}).sort_values("VIF", ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
colors_vif = ["#E74C3C" if v > 10 else "#F39C12" if v > 5 else "#2ECC71" for v in vif_df["VIF"]]
ax.barh(range(len(vif_df)), vif_df["VIF"].clip(0, 50), color=colors_vif, edgecolor="white")
ax.set_yticks(range(len(vif_df)))
ax.set_yticklabels(vif_df["Feature"])
ax.set_xlabel("Variance Inflation Factor")
ax.axvline(x=5, color="orange", linestyle="--", linewidth=1, label="Moderate (VIF=5)")
ax.axvline(x=10, color="red", linestyle="--", linewidth=1, label="Severe (VIF=10)")
ax.set_title("Multicollinearity Assessment: Variance Inflation Factors\n"
             "Red > 10, Orange > 5, Green < 5", fontweight="bold")
ax.legend(fontsize=8)

for i, (_, row) in enumerate(vif_df.iterrows()):
    val = row["VIF"]
    label = f"{val:.1f}" if val < 100 else f"{val:.0f}"
    ax.text(min(val, 50) + 0.3, i, label, va="center", fontsize=7)

plt.tight_layout()
plt.show()

print("\n--- VIF Summary ---")
print(vif_df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
severe = vif_df[vif_df["VIF"] > 10]
if len(severe) > 0:
    print(f"\n{len(severe)} features with VIF > 10 (severe multicollinearity):")
    for _, row in severe.iterrows():
        print(f"  {row['Feature']}: VIF = {row['VIF']:.1f}")
    print("\nNote: High VIF among BILL_AMT is expected (r > 0.95 between adjacent months).")
    print("The transformer attention mechanism handles this naturally by learning")
    print("to focus on temporal changes rather than absolute values.")


### 2.6 Mutual Information Analysis

Mutual information $I(X; Y)$ captures **non-linear** associations that Pearson correlation misses. Unlike $|r_{pb}|$, MI detects arbitrary dependency structure --- critical for features like PAY status where the default-rate function is non-monotonic.


In [ ]:
# --- Mutual Information ---
from sklearn.feature_selection import mutual_info_classif

X_mi = df[ALL_FEATURES].copy()
y_mi = df[TARGET].values

# Identify discrete features for MI computation
discrete_mask = [col in CATEGORICALS + PAY_STATUS for col in ALL_FEATURES]

mi_scores = mutual_info_classif(X_mi, y_mi, discrete_features=discrete_mask, random_state=42, n_neighbors=5)
mi_df = pd.DataFrame({"Feature": ALL_FEATURES, "MI": mi_scores}).sort_values("MI", ascending=True)

# Also get point-biserial for comparison
from scipy.stats import pointbiserialr
pb_scores = {}
for col in ALL_FEATURES:
    if col in CATEGORICALS:
        pb_scores[col] = cramers_v(df[col], df[TARGET])
    else:
        r, _ = pointbiserialr(df[TARGET], df[col])
        pb_scores[col] = abs(r)

mi_df["Linear (|r| or V)"] = mi_df["Feature"].map(pb_scores)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# (a) MI ranking
ax = axes[0]
ax.barh(range(len(mi_df)), mi_df["MI"], color="#6C5CE7", edgecolor="white", alpha=0.8)
ax.set_yticks(range(len(mi_df)))
ax.set_yticklabels(mi_df["Feature"], fontsize=8)
ax.set_xlabel("Mutual Information (nats)")
ax.set_title("(a) Non-Linear Association\n(Mutual Information)", fontweight="bold")

# (b) MI vs Linear comparison
ax = axes[1]
ax.scatter(mi_df["Linear (|r| or V)"], mi_df["MI"], s=50, c="#E17055", edgecolors="white", zorder=5)
for _, row in mi_df.iterrows():
    ax.annotate(row["Feature"], (row["Linear (|r| or V)"], row["MI"]),
                fontsize=6, ha="left", va="bottom", xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Linear Association (|r_pb| or Cramer's V)")
ax.set_ylabel("Mutual Information (nats)")
ax.set_title("(b) Linear vs Non-Linear Association\n(features above diagonal have hidden non-linear signal)",
             fontweight="bold")

# Add diagonal reference
max_val = max(mi_df["MI"].max(), mi_df["Linear (|r| or V)"].max())
ax.plot([0, max_val], [0, max_val], "k--", alpha=0.3, linewidth=1)

fig.suptitle("Feature Importance: Mutual Information vs Linear Measures",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Print ranking comparison
print("\n--- MI vs Linear Association Ranking ---")
print(f"{'Rank':<6} {'Feature (MI)':<15} {'MI':>8} {'Feature (Linear)':<15} {'|r|/V':>8}")
print("-" * 55)
mi_ranked = mi_df.sort_values("MI", ascending=False)
lin_ranked = mi_df.sort_values("Linear (|r| or V)", ascending=False)
for i in range(min(10, len(mi_ranked))):
    mi_row = mi_ranked.iloc[i]
    lin_row = lin_ranked.iloc[i]
    print(f"{i+1:<6} {mi_row['Feature']:<15} {mi_row['MI']:>8.4f} {lin_row['Feature']:<15} {lin_row['Linear (|r| or V)']:>8.4f}")


<a id="3-target-variable-analysis"></a>
## 3. Target Variable Analysis

The target variable encodes whether a client defaulted on their payment in October 2005 (1 = default, 0 = no default). Characterising the class imbalance is critical: it determines the loss function, evaluation metrics, and sampling strategy.

**Implication.** A naive majority-class classifier achieves ~78% accuracy. We must use:
- **Class-weighted BCE** or **focal loss** (Lin et al., 2017) during training
- **AUC-ROC**, **F1-score**, and **precision-recall curves** for evaluation (not accuracy)
- **Stratified splitting** to preserve the 22.1% default rate across train/val/test

In [ ]:
# --- Figure 1: Class distribution ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df[TARGET].value_counts().sort_index()
labels = ["No Default", "Default"]
colors = [C_ND, C_D]

# (a) Bar chart
ax = axes[0]
bars = ax.bar(labels, counts.values, color=colors, edgecolor="white", linewidth=1.5)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f"{count:,}", ha="center", va="bottom", fontweight="bold", fontsize=10)
ax.set_ylabel("Count")
ax.set_title("(a) Class Counts")
ax.set_ylim(0, counts.max() * 1.15)

# (b) Pie chart
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    counts.values, labels=labels, colors=colors, autopct="%1.1f%%",
    startangle=90, textprops={"fontsize": 10},
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
for t in autotexts:
    t.set_fontweight("bold")
ax.set_title("(b) Class Proportions")

imbalance = counts[0] / counts[1]
fig.suptitle(
    f"Target Distribution --- Imbalance Ratio {imbalance:.1f}:1",
    fontsize=13, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

print(f"\nImbalance ratio: {imbalance:.2f}:1")
print(f"Default rate:    {counts[1] / counts.sum():.4f}")
print(f"Baseline accuracy (predict majority): {counts[0] / counts.sum():.4f}")

In [ ]:
# --- Empirical Cumulative Distribution Functions (ECDFs) for key features ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, feat in enumerate(["LIMIT_BAL", "AGE", "PAY_AMT1"]):
    ax = axes[idx]
    for target_val, label, color in [(0, "No Default", C_ND), (1, "Default", C_D)]:
        vals = np.sort(df[df[TARGET] == target_val][feat].values)
        ecdf = np.arange(1, len(vals) + 1) / len(vals)
        ax.step(vals, ecdf, color=color, label=label, linewidth=1.5, where="post")

    # KS statistic annotation
    d0 = df[df[TARGET] == 0][feat]
    d1 = df[df[TARGET] == 1][feat]
    ks_stat, ks_p = stats.ks_2samp(d0, d1)
    ax.text(0.95, 0.05, f"KS = {ks_stat:.3f}\np < 0.001" if ks_p < 0.001 else f"KS = {ks_stat:.3f}\np = {ks_p:.3f}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.5))
    ax.set_xlabel(feat)
    ax.set_ylabel("ECDF")
    ax.set_title(f"({chr(97+idx)}) {feat}")
    ax.legend(fontsize=7)

fig.suptitle("Empirical CDFs --- Two-Sample Kolmogorov-Smirnov Tests",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Violin plots: PAY_AMT distributions by default status ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, col in enumerate(PAY_AMT):
    ax = axes[idx // 3][idx % 3]
    month = MONTH_LABELS[idx]

    # Cap at 99th percentile for readability
    cap = df[col].quantile(0.99)
    plot_data = df[[col, TARGET]].copy()
    plot_data[col] = plot_data[col].clip(0, cap)
    plot_data["Status"] = plot_data[TARGET].map({0: "No Default", 1: "Default"})

    parts = ax.violinplot(
        [plot_data[plot_data[TARGET] == 0][col].values,
         plot_data[plot_data[TARGET] == 1][col].values],
        positions=[0, 1], showmeans=True, showmedians=True, widths=0.7
    )
    for i, pc in enumerate(parts["bodies"]):
        pc.set_facecolor([C_ND, C_D][i])
        pc.set_alpha(0.6)
    parts["cmeans"].set_color("black")
    parts["cmedians"].set_color("white")

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["No Default", "Default"])
    ax.set_ylabel("Amount (NT$)")
    ax.set_title(f"{col} ({month})")

    # Annotate medians
    for t, label in [(0, "No Default"), (1, "Default")]:
        med = df[df[TARGET] == t][col].median()
        ax.text(t, cap * 0.95, f"med={med:,.0f}", ha="center", fontsize=7, style="italic")

fig.suptitle("Payment Amount Distributions by Default Status (Violin Plots, capped at P99)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Default rate heatmap: AGE x LIMIT_BAL (2D binned) ---
fig, ax = plt.subplots(figsize=(10, 7))

age_bins = pd.cut(df["AGE"], bins=[20, 25, 30, 35, 40, 45, 50, 55, 60, 80],
                  labels=["21-25", "26-30", "31-35", "36-40", "41-45", "46-50", "51-55", "56-60", "61+"])
limit_bins = pd.cut(df["LIMIT_BAL"], bins=[0, 50000, 100000, 200000, 300000, 500000, 1000000],
                    labels=["<50k", "50-100k", "100-200k", "200-300k", "300-500k", "500k+"])

heatmap = df.assign(AGE_BIN=age_bins, LIMIT_BIN=limit_bins).pivot_table(
    values=TARGET, index="LIMIT_BIN", columns="AGE_BIN", aggfunc=["mean", "count"]
)

rates = heatmap["mean"]
counts = heatmap["count"]

sns.heatmap(rates, ax=ax, cmap="YlOrRd", annot=True, fmt=".1%",
            linewidths=0.5, vmin=0.10, vmax=0.40,
            cbar_kws={"label": "Default Rate"})

# Overlay sample sizes
for i in range(rates.shape[0]):
    for j in range(rates.shape[1]):
        n = counts.iloc[i, j]
        if not np.isnan(n):
            ax.text(j + 0.5, i + 0.75, f"n={int(n)}", ha="center", va="center",
                    fontsize=6, color="gray", style="italic")

ax.set_xlabel("Age Group")
ax.set_ylabel("Credit Limit Bracket")
ax.set_title("Default Rate by Age x Credit Limit\n(annotated with sample sizes)",
             fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- Radar chart: Normalised defaulter vs non-defaulter profile ---
from matplotlib.patches import FancyBboxPatch

# Compute normalised profile means for key features
radar_features = ["LIMIT_BAL", "AGE", "PAY_0", "BILL_AMT1", "PAY_AMT1"]
radar_labels = ["Credit\nLimit", "Age", "Recent\nDelinquency", "Recent\nBill", "Recent\nPayment"]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

angles = np.linspace(0, 2 * np.pi, len(radar_features), endpoint=False).tolist()
angles += angles[:1]

for target_val, label, color in [(0, "No Default", C_ND), (1, "Default", C_D)]:
    subset = df[df[TARGET] == target_val]
    values = []
    for feat in radar_features:
        # Normalise to [0, 1] range based on full dataset
        v = (subset[feat].mean() - df[feat].min()) / (df[feat].max() - df[feat].min())
        values.append(v)
    values += values[:1]

    ax.plot(angles, values, "o-", linewidth=2, color=color, label=label, markersize=5)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=9)
ax.set_ylim(0, 0.5)
ax.set_title("Normalised Feature Profile: Defaulters vs Non-Defaulters",
             fontsize=12, fontweight="bold", y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.2, 1.1))
plt.tight_layout()
plt.show()

<a id="4-univariate-feature-analysis"></a>
## 4. Univariate Feature Analysis

### 4.1 Categorical Features

For each categorical feature we compute:
- **Default rate per category** with 95% Wilson binomial confidence intervals
- **Chi-squared test of independence** ($H_0$: feature and default are independent)
- **Cramer's V** as the effect size measure:

$$V = \sqrt{\frac{\chi^2}{n \cdot (\min(r, c) - 1)}}$$

where $r$ and $c$ are the number of rows and columns in the contingency table. Convention: $V < 0.1$ negligible, $0.1$--$0.3$ small, $0.3$--$0.5$ medium, $> 0.5$ large.

In [ ]:
cat_features = {
    "SEX": {1: "Male", 2: "Female"},
    "EDUCATION": {1: "Grad School", 2: "University", 3: "High School", 4: "Others"},
    "MARRIAGE": {1: "Married", 2: "Single", 3: "Others"},
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (feat, label_map) in enumerate(cat_features.items()):
    ax = axes[idx]

    grouped = df.groupby(feat)[TARGET].agg(["mean", "count", "sum"])
    grouped.index = grouped.index.map(label_map)

    # Chi-squared test and Cramer's V
    ct = pd.crosstab(df[feat], df[TARGET])
    chi2, p_val, dof, _ = chi2_contingency(ct)
    v = cramers_v(df[feat], df[TARGET])

    # Wilson confidence intervals
    x = np.arange(len(grouped))
    rates = grouped["mean"].values
    ns = grouped["count"].values
    ci_low, ci_high = zip(*[wilson_ci(r, n) for r, n in zip(rates, ns)])

    # Stacked bars
    ax.bar(x, 1 - rates, color=C_ND, label="No Default", edgecolor="white", linewidth=0.5)
    ax.bar(x, rates, bottom=1 - rates, color=C_D, label="Default", edgecolor="white", linewidth=0.5)

    # Error bars for default rate
    ax.errorbar(x, rates + (1 - rates), yerr=[np.array(rates) - np.array(ci_low),
                np.array(ci_high) - np.array(rates)],
                fmt="none", ecolor="black", capsize=3, linewidth=1)

    for i, (rate, n) in enumerate(zip(rates, ns)):
        ax.text(i, 1.05, f"{rate:.1%}\n(n={n:,})", ha="center", fontsize=7, va="bottom")

    ax.set_xticks(x)
    ax.set_xticklabels(grouped.index, rotation=15, ha="right")
    ax.set_ylim(0, 1.2)
    ax.set_ylabel("Proportion")
    p_str = "p < 0.001" if p_val < 0.001 else f"p = {p_val:.3f}"
    ax.set_title(f"{feat}\nX2={chi2:.1f}, {p_str}, V={v:.3f}")
    if idx == 0:
        ax.legend(loc="lower left", frameon=True, framealpha=0.9)

fig.suptitle("Categorical Features: Default Rate by Category (with 95% Wilson CI)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print("\n--- Categorical Feature Association Summary ---")
print(f"{'Feature':<15} {'Chi2':>8} {'p-value':>12} {'Cramers V':>10} {'Effect':>12}")
print("-" * 60)
for feat in CATEGORICALS:
    chi2, p, _, _ = chi2_contingency(pd.crosstab(df[feat], df[TARGET]))
    v = cramers_v(df[feat], df[TARGET])
    effect = "negligible" if v < 0.1 else "small" if v < 0.3 else "medium"
    print(f"{feat:<15} {chi2:>8.1f} {'<0.001' if p < 0.001 else f'{p:.4f}':>12} {v:>10.4f} {effect:>12}")


### 4.2 Numerical Features (LIMIT_BAL, AGE)

For continuous features stratified by default status, we apply:
- **Two-sample Kolmogorov--Smirnov test** ($H_0$: the two samples are drawn from the same distribution)
- **Mann--Whitney U test** (non-parametric rank-sum test, appropriate for skewed distributions)
- **Rank-biserial correlation** as effect size: $r_{rb} = 1 - \frac{2U}{n_0 n_1}$
- **Cohen's d** for standardised mean difference: $d = \frac{\bar{x}_0 - \bar{x}_1}{s_{\text{pooled}}}$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for idx, feat in enumerate(["LIMIT_BAL", "AGE"]):
    ax = axes[idx]
    d0 = df[df[TARGET] == 0][feat]
    d1 = df[df[TARGET] == 1][feat]

    # Statistical tests
    U, p_mw = mannwhitneyu(d0, d1, alternative="two-sided")
    r_rb = 1 - (2 * U) / (len(d0) * len(d1))
    ks_stat, p_ks = stats.ks_2samp(d0, d1)
    d_cohen = cohens_d(d0, d1)

    # Histogram + KDE
    ax.hist(d0, bins=50, density=True, alpha=0.35, color=C_ND, label="No Default")
    ax.hist(d1, bins=50, density=True, alpha=0.35, color=C_D, label="Default")
    d0.plot.kde(ax=ax, color=C_ND, linewidth=2)
    d1.plot.kde(ax=ax, color=C_D, linewidth=2)

    # Vertical lines for means
    ax.axvline(d0.mean(), color=C_ND, linestyle="--", linewidth=1, alpha=0.7)
    ax.axvline(d1.mean(), color=C_D, linestyle="--", linewidth=1, alpha=0.7)

    p_mw_str = "p < 0.001" if p_mw < 0.001 else f"p = {p_mw:.4f}"
    ax.set_title(f"{feat}\nMW {p_mw_str}, r_rb={r_rb:.3f}, d={d_cohen:.3f}\nKS={ks_stat:.3f}")
    ax.set_ylabel("Density")
    ax.legend()

    if feat == "LIMIT_BAL":
        ax.set_xlabel("Credit Limit (NT$)")
    else:
        ax.set_xlabel("Age (years)")

fig.suptitle("Numerical Feature Distributions by Default Status",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Quantile comparison
print("\n--- LIMIT_BAL Quantile Comparison ---")
quantiles = [0.10, 0.25, 0.50, 0.75, 0.90]
print(f"{'Quantile':<12} {'No Default':>12} {'Default':>12} {'Ratio':>8}")
print("-" * 48)
for q in quantiles:
    v0 = df[df[TARGET] == 0]["LIMIT_BAL"].quantile(q)
    v1 = df[df[TARGET] == 1]["LIMIT_BAL"].quantile(q)
    print(f"{q:<12.2f} {v0:>12,.0f} {v1:>12,.0f} {v0/v1:>8.2f}")


In [ ]:
# --- Box plots: BILL_AMT across all 6 months, split by default ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, col in enumerate(BILL_AMT):
    ax = axes[idx // 3][idx % 3]
    month = MONTH_LABELS[idx]

    cap = df[col].quantile(0.98)
    data_nd = df[df[TARGET] == 0][col].clip(df[col].quantile(0.02), cap)
    data_d = df[df[TARGET] == 1][col].clip(df[col].quantile(0.02), cap)

    bp = ax.boxplot([data_nd, data_d], positions=[0, 1], widths=0.5,
                    patch_artist=True, showfliers=False,
                    medianprops=dict(color="white", linewidth=2))
    bp["boxes"][0].set_facecolor(C_ND)
    bp["boxes"][0].set_alpha(0.6)
    bp["boxes"][1].set_facecolor(C_D)
    bp["boxes"][1].set_alpha(0.6)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["No Default", "Default"])
    ax.set_ylabel("Amount (NT$)")
    ax.set_title(f"{col} ({month})")

    # Mann-Whitney test
    U, p_val = mannwhitneyu(data_nd, data_d)
    p_str = "p < 0.001" if p_val < 0.001 else f"p = {p_val:.3f}"
    ax.text(0.5, 0.95, p_str, transform=ax.transAxes, ha="center", fontsize=7,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

fig.suptitle("Bill Amount Distributions by Default Status (Box Plots, P2-P98 range)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 4.3 PAY Status Semantic Analysis

The PAY features (PAY_0 through PAY_6) encode repayment status for each of the 6 months. Their value semantics are non-trivial:

| Value | Meaning |
|:---:|:---|
| -2 | No bill (no consumption that month) |
| -1 | Paid in full |
| 0 | Revolving credit (minimum payment made) |
| 1 | Payment delay of 1 month |
| 2--8 | Payment delay of 2--8 months |

This dual-zone structure --- **categorical** for $\{-2, -1, 0\}$ and **ordinal** for $\{1, 2, \ldots, 8\}$ --- is the single most important observation for tokenisation design. A flat embedding table conflates the semantic discontinuity between "paid in full" ($-1$) and "1 month late" ($1$), which have fundamentally different risk implications.

In [ ]:
# --- Figure 4: PAY status semantic analysis ---
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1], hspace=0.35, wspace=0.3)

# (a) PAY_0 distribution by default status
ax = fig.add_subplot(gs[0, 0])
pay_vals = sorted(df["PAY_0"].unique())
d0_counts = df[df[TARGET] == 0]["PAY_0"].value_counts().reindex(pay_vals, fill_value=0)
d1_counts = df[df[TARGET] == 1]["PAY_0"].value_counts().reindex(pay_vals, fill_value=0)

x = np.arange(len(pay_vals))
w = 0.35
ax.bar(x - w / 2, d0_counts.values, w, color=C_ND, label="No Default", edgecolor="white")
ax.bar(x + w / 2, d1_counts.values, w, color=C_D, label="Default", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(pay_vals)
ax.set_xlabel("PAY_0 Value")
ax.set_ylabel("Count")
ax.set_title("(a) PAY_0 Distribution by Default Status")
ax.legend()

# (b) Default rate by PAY_0 value with Wilson CI
ax = fig.add_subplot(gs[0, 1])
default_rates = df.groupby("PAY_0")[TARGET].mean().reindex(pay_vals)
counts_per_val = df.groupby("PAY_0").size().reindex(pay_vals)

ci_lo, ci_hi = zip(*[wilson_ci(r, n) for r, n in zip(default_rates.values, counts_per_val.values)])
ci_lo, ci_hi = np.array(ci_lo), np.array(ci_hi)

bars = ax.bar(x, default_rates.values,
              color=[C_D if r > 0.5 else C_ND for r in default_rates],
              edgecolor="white", alpha=0.8)
ax.errorbar(x, default_rates.values,
            yerr=[default_rates.values - ci_lo, ci_hi - default_rates.values],
            fmt="none", ecolor="black", capsize=3, linewidth=1)

for i, (rate, n) in enumerate(zip(default_rates.values, counts_per_val.values)):
    ax.text(i, rate + 0.04, f"{rate:.1%}\n(n={n:,})", ha="center", fontsize=6, va="bottom")

ax.axhline(y=df[TARGET].mean(), color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.text(len(pay_vals) - 1, df[TARGET].mean() + 0.01, "Overall rate", fontsize=7, color="gray", ha="right")
ax.set_xticks(x)
ax.set_xticklabels(pay_vals)
ax.set_xlabel("PAY_0 Value")
ax.set_ylabel("Default Rate")
ax.set_title("(b) Default Rate by PAY_0 (with 95% Wilson CI)")

# Semantic zone shading
ax.axvspan(-0.5, 2.5, alpha=0.08, color="green")
ax.axvspan(2.5, len(pay_vals) - 0.5, alpha=0.08, color="red")
ax.text(1, ax.get_ylim()[1] * 0.92, "Categorical\nzone (-2,-1,0)",
        ha="center", fontsize=7, color="green", fontweight="bold")
ax.text(len(pay_vals) - 3, ax.get_ylim()[1] * 0.92, "Ordinal\ndelinquency",
        ha="center", fontsize=7, color="red", fontweight="bold")

# (c) PAY heatmap across 6 months
ax = fig.add_subplot(gs[1, :])
all_pay_vals = sorted(set().union(*[set(df[c].unique()) for c in PAY_STATUS]))

heatmap_data = []
row_labels = []
for target_val, label in [(0, "No Default"), (1, "Default")]:
    subset = df[df[TARGET] == target_val]
    for col, month in zip(PAY_STATUS, MONTH_LABELS):
        proportions = subset[col].value_counts(normalize=True).reindex(all_pay_vals, fill_value=0)
        heatmap_data.append(proportions.values)
        row_labels.append(f"{label} -- {month}")

heatmap_df = pd.DataFrame(heatmap_data, index=row_labels, columns=all_pay_vals)
sns.heatmap(heatmap_df, ax=ax, cmap="YlOrRd", annot=True, fmt=".2f",
            linewidths=0.5, cbar_kws={"label": "Proportion"}, annot_kws={"size": 6})
ax.set_xlabel("PAY Status Value")
ax.set_title("(c) PAY Status Distribution Across Months -- Defaulters vs Non-Defaulters")

fig.suptitle("PAY Feature Semantic Analysis -- Motivating Tokenisation Design",
             fontsize=14, fontweight="bold", y=1.01)
plt.show()

<a id="5-bivariate-feature-target-associations"></a>
## 5. Bivariate Feature--Target Associations

We rank all 23 features by their univariate association with the binary target using two complementary measures:

- **Point-biserial correlation** $|r_{pb}|$ for numerical and ordinal features --- equivalent to Pearson's $r$ between a continuous and a dichotomous variable
- **Cramer's V** for categorical features --- normalised chi-squared

This ranking provides a prior on which features the transformer's attention heads should weight most heavily. It also guides feature selection for the Random Forest baseline.

In [ ]:
# --- Figure 8: Feature-target association ranking ---
associations = {}

for col in NUMERICALS:
    r, p = pointbiserialr(df[TARGET], df[col])
    associations[col] = {"strength": abs(r), "metric": "|r_pb|", "p_value": p, "raw_r": r}

for col in CATEGORICALS:
    v = cramers_v(df[col], df[TARGET])
    associations[col] = {"strength": v, "metric": "Cramers V", "p_value": np.nan, "raw_r": np.nan}

for col in PAY_STATUS:
    r, p = pointbiserialr(df[TARGET], df[col])
    associations[col] = {"strength": abs(r), "metric": "|r_pb|", "p_value": p, "raw_r": r}

assoc_df = pd.DataFrame(associations).T.sort_values("strength", ascending=True)

fig, ax = plt.subplots(figsize=(8, 10))

color_map = {
    "PAY": "#E17055", "CAT": "#00B894", "BILL": "#6C5CE7",
    "PAY_AMT": "#FDCB6E", "DEMO": "#74B9FF"
}
colors = []
for feat in assoc_df.index:
    if feat in PAY_STATUS:
        colors.append(color_map["PAY"])
    elif feat in CATEGORICALS:
        colors.append(color_map["CAT"])
    elif feat in BILL_AMT:
        colors.append(color_map["BILL"])
    elif feat in PAY_AMT:
        colors.append(color_map["PAY_AMT"])
    else:
        colors.append(color_map["DEMO"])

ax.barh(range(len(assoc_df)), assoc_df["strength"], color=colors, edgecolor="white")
ax.set_yticks(range(len(assoc_df)))
ax.set_yticklabels(assoc_df.index)
ax.set_xlabel("Association Strength")
ax.set_title("Feature-Target Association Strength\n"
             "(Point-biserial |r| for numerical, Cramer's V for categorical)",
             fontweight="bold")

legend_elements = [
    Patch(facecolor="#E17055", label="PAY Status"),
    Patch(facecolor="#74B9FF", label="Demographic (num.)"),
    Patch(facecolor="#00B894", label="Demographic (cat.)"),
    Patch(facecolor="#6C5CE7", label="Bill Amount"),
    Patch(facecolor="#FDCB6E", label="Payment Amount"),
]
ax.legend(handles=legend_elements, loc="lower right", frameon=True)

plt.tight_layout()
plt.show()

# Print ranked table
print("\n--- Feature-Target Association Ranking ---")
print(f"{'Rank':<6} {'Feature':<15} {'Metric':<12} {'Strength':>10} {'Direction':>10}")
print("-" * 55)
for rank, (feat, row) in enumerate(assoc_df.iloc[::-1].iterrows(), 1):
    direction = ""
    if not np.isnan(row["raw_r"]):
        direction = "+" if row["raw_r"] > 0 else "-"
    print(f"{rank:<6} {feat:<15} {row['metric']:<12} {row['strength']:>10.4f} {direction:>10}")

<a id="6-temporal-structure-analysis"></a>
## 6. Temporal Structure Analysis

This is the most consequential analysis section. The dataset contains **6 monthly snapshots** (April--September 2005) for three feature groups: repayment status, bill amounts, and payment amounts. If defaulters and non-defaulters exhibit diverging temporal trajectories, it justifies using a **sequence model** (transformer with positional attention) rather than treating features as an unordered bag.

We compute:
- **Mean trajectories** with 95% confidence intervals of the mean
- **Spearman rank correlation** between temporal position and feature value (monotonic trend strength)
- **Autocorrelation analysis** to quantify temporal dependency structure

In [ ]:
# --- Figure 5: Temporal trajectories with 95% CI ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

feature_groups = [
    ("Repayment Status", PAY_STATUS, "Mean PAY Value"),
    ("Bill Amount", BILL_AMT, "Mean Amount (NT$)"),
    ("Payment Amount", PAY_AMT, "Mean Amount (NT$)"),
]

for idx, (title, cols, ylabel) in enumerate(feature_groups):
    ax = axes[idx]

    for target_val, label, color in [(0, "No Default", C_ND), (1, "Default", C_D)]:
        subset = df[df[TARGET] == target_val][cols]
        means = subset.mean()
        sems = subset.std() / np.sqrt(len(subset))

        ax.plot(MONTH_LABELS, means.values, marker="o", color=color,
                label=label, linewidth=2, markersize=5)
        ax.fill_between(MONTH_LABELS,
                        means.values - 1.96 * sems.values,
                        means.values + 1.96 * sems.values,
                        alpha=0.15, color=color)

    ax.set_xlabel("Month (2005)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"({chr(97 + idx)}) {title}")
    ax.legend()
    ax.invert_xaxis()

fig.suptitle(
    "Temporal Trajectories: Diverging Patterns Between Defaulters and Non-Defaulters\n"
    "(Shaded regions = 95% CI of the mean)",
    fontsize=13, fontweight="bold", y=1.05,
)
plt.tight_layout()
plt.show()

# --- Spearman trend test per group ---
print("\n--- Temporal Trend Analysis (Spearman rho across 6 months) ---")
print(f"{'Group':<20} {'Class':<15} {'rho':>8} {'p-value':>12} {'Interpretation':<25}")
print("-" * 80)
months = np.arange(6)
for title, cols, _ in feature_groups:
    for target_val, label in [(0, "No Default"), (1, "Default")]:
        subset = df[df[TARGET] == target_val][cols]
        means = subset.mean().values
        rho, p = spearmanr(months, means)
        interp = "strong monotonic" if abs(rho) > 0.8 else "moderate" if abs(rho) > 0.5 else "weak"
        direction = "increasing" if rho > 0 else "decreasing"
        print(f"{title:<20} {label:<15} {rho:>8.3f} {p:>12.4f} {interp} {direction}")

In [ ]:
# --- Stacked area chart: PAY status composition over 6 months ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

pay_labels = {-2: "No bill", -1: "Paid in full", 0: "Revolving", 1: "1mo late", 2: "2mo late", 3: "3+ mo late"}
pay_colors = ["#2ecc71", "#27ae60", "#f39c12", "#e74c3c", "#c0392b", "#8e44ad"]

for target_val, label, ax in [(0, "No Default", axes[0]), (1, "Default", axes[1])]:
    subset = df[df[TARGET] == target_val]

    proportions = {}
    for pay_cat, pay_label in pay_labels.items():
        props = []
        for col in PAY_STATUS:
            if pay_cat == 3:
                props.append((subset[col] >= 3).mean())
            else:
                props.append((subset[col] == pay_cat).mean())
        proportions[pay_label] = props

    prop_df = pd.DataFrame(proportions, index=MONTH_LABELS)
    prop_df.plot.area(ax=ax, stacked=True, color=pay_colors, alpha=0.8, linewidth=0.5)
    ax.set_xlabel("Month (2005)")
    ax.set_ylabel("Proportion")
    ax.set_title(f"{label} (n={len(subset):,})")
    ax.set_ylim(0, 1)
    ax.invert_xaxis()
    ax.legend(fontsize=7, loc="upper left")

fig.suptitle("PAY Status Composition Over Time --- Stacked Area Chart",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Ridge plot: BILL_AMT distributions across 6 months ---
fig, axes = plt.subplots(6, 1, figsize=(10, 10), sharex=True)

for idx, (col, month) in enumerate(zip(BILL_AMT, MONTH_LABELS)):
    ax = axes[idx]
    cap = df[col].quantile(0.99)
    for target_val, label, color in [(0, "No Default", C_ND), (1, "Default", C_D)]:
        vals = df[df[TARGET] == target_val][col].clip(-50000, cap)
        vals.plot.kde(ax=ax, color=color, linewidth=1.5, label=label)
        ax.fill_between(ax.lines[-1].get_xdata(), ax.lines[-1].get_ydata(),
                        alpha=0.2, color=color)

    ax.set_ylabel(month, rotation=0, ha="right", fontsize=10, fontweight="bold")
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)
    if idx == 0:
        ax.legend(fontsize=8)
    if idx < 5:
        ax.spines["bottom"].set_visible(False)
        ax.tick_params(bottom=False)

axes[-1].set_xlabel("Bill Amount (NT$)")
fig.suptitle("BILL_AMT Distributions Across 6 Months (Ridge Plot)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 9: BILL_AMT autocorrelation structure ---
bill_corr = df[BILL_AMT].corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) Cross-month correlation matrix
ax = axes[0]
sns.heatmap(bill_corr, ax=ax, cmap="YlOrRd", annot=True, fmt=".3f",
            square=True, linewidths=0.5, vmin=0.5, vmax=1,
            xticklabels=MONTH_LABELS, yticklabels=MONTH_LABELS)
ax.set_title("(a) BILL_AMT Cross-Month Correlation")

# (b) Autocorrelation decay by lag, split by default status
ax = axes[1]
for target_val, label, color in [(0, "No Default", C_ND), (1, "Default", C_D)]:
    subset = df[df[TARGET] == target_val]
    lag_corrs = []
    lag_cis = []
    for lag in range(1, 6):
        pairs = []
        for i in range(6 - lag):
            r = subset[BILL_AMT[i]].corr(subset[BILL_AMT[i + lag]])
            pairs.append(r)
        lag_corrs.append(np.mean(pairs))
        lag_cis.append(np.std(pairs) / np.sqrt(len(pairs)))

    lags = list(range(1, 6))
    ax.errorbar(lags, lag_corrs, yerr=[1.96 * c for c in lag_cis],
                marker="o", color=color, label=label, linewidth=2,
                markersize=5, capsize=3)

ax.set_xlabel("Lag (months)")
ax.set_ylabel("Mean Pearson Correlation")
ax.set_title("(b) Autocorrelation Decay by Default Status")
ax.legend()
ax.set_ylim(0.5, 1.0)

fig.suptitle("Bill Amount Temporal Structure -- Motivating Self-Attention over Time Steps",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Numerical summary
print("\n--- BILL_AMT Autocorrelation at Lag 1 ---")
for target_val, label in [(0, "No Default"), (1, "Default")]:
    subset = df[df[TARGET] == target_val]
    lag1_corrs = [subset[BILL_AMT[i]].corr(subset[BILL_AMT[i+1]]) for i in range(5)]
    print(f"  {label}: mean r = {np.mean(lag1_corrs):.4f} (range {min(lag1_corrs):.4f} -- {max(lag1_corrs):.4f})")

<a id="7-correlation-and-multicollinearity"></a>
## 7. Correlation and Multicollinearity

A full $24 \times 24$ Pearson correlation matrix reveals the internal dependency structure. Key patterns to identify:

- **Redundancy**: If BILL_AMT features are highly correlated ($r > 0.9$), the transformer's attention mechanism should learn to focus on *changes* rather than absolute values.
- **Multicollinearity**: Features with $|r| > 0.8$ may cause instability in linear models but are handled naturally by tree ensembles and attention mechanisms.
- **Target correlation**: Which features correlate most with DEFAULT?

In [ ]:
# --- Figure 7: Full correlation heatmap ---
numeric_cols = [c for c in ALL_FEATURES if c in df.select_dtypes(include=[np.number]).columns]
corr = df[numeric_cols + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap=CMAP_DIV, center=0, ax=ax,
            square=True, linewidths=0.3,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"},
            annot=True, fmt=".2f", annot_kws={"size": 5},
            vmin=-1, vmax=1)
ax.set_title("Feature Correlation Matrix (Lower Triangle)\n"
             "Strong BILL_AMT autocorrelation; PAY features most correlated with target",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# --- Identify high-correlation pairs ---
print("\n--- Feature Pairs with |r| > 0.80 ---")
print(f"{'Feature A':<15} {'Feature B':<15} {'Pearson r':>10}")
print("-" * 42)
seen = set()
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        r_val = corr.iloc[i, j]
        if abs(r_val) > 0.80:
            a, b = corr.columns[i], corr.columns[j]
            if (a, b) not in seen:
                seen.add((a, b))
                print(f"{a:<15} {b:<15} {r_val:>10.4f}")

# --- Target correlation ranking ---
print("\n--- Correlation with DEFAULT (top 10) ---")
target_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False)
for feat, r_val in target_corr.head(10).items():
    sign = "+" if corr.loc[feat, TARGET] > 0 else "-"
    print(f"  {feat:<15} {sign}{r_val:.4f}")

In [ ]:
# --- Pair plot: Top 4 most predictive features ---
top_features = ["PAY_0", "PAY_2", "LIMIT_BAL", "PAY_AMT1"]
sample = df[top_features + [TARGET]].sample(n=3000, random_state=42)
sample["Class"] = sample[TARGET].map({0: "No Default", 1: "Default"})

g = sns.pairplot(sample, vars=top_features, hue="Class",
                 palette={"No Default": C_ND, "Default": C_D},
                 plot_kws={"alpha": 0.15, "s": 8},
                 diag_kws={"alpha": 0.5},
                 height=2.2, corner=True)
g.fig.suptitle("Pair Plot: Top 4 Predictive Features (sampled n=3,000)",
               fontsize=13, fontweight="bold", y=1.02)
plt.show()

In [ ]:
# --- Clustered correlation: PAY features only (defaulters vs non-defaulters) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for target_val, label, ax in [(0, "No Default", axes[0]), (1, "Default", axes[1])]:
    subset = df[df[TARGET] == target_val]
    pay_corr = subset[PAY_STATUS].corr()

    sns.heatmap(pay_corr, ax=ax, cmap="YlOrRd", annot=True, fmt=".2f",
                square=True, linewidths=0.5, vmin=0, vmax=1,
                xticklabels=MONTH_LABELS, yticklabels=MONTH_LABELS)
    ax.set_title(f"{label} (n={len(subset):,})")

fig.suptitle("PAY Feature Correlation Structure: Defaulters vs Non-Defaulters\n"
             "Defaulters show weaker inter-month correlation (less stable behaviour)",
             fontsize=12, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

<a id="8-derived-risk-indicators"></a>
## 8. Derived Risk Indicators

Two domain-motivated risk indicators that synthesise raw features:

### 8.1 Credit Utilisation Ratio
$$\text{UTIL}_t = \frac{\text{BILL\_AMT}_t}{\text{LIMIT\_BAL}}$$

Clients near or above their credit limit ($\text{UTIL} \geq 1.0$) are at elevated risk. Over-utilisation indicates financial distress.

### 8.2 Repayment Ratio
$$\text{REPAY}_t = \frac{\text{PAY\_AMT}_t}{|\text{BILL\_AMT}_t|}$$

Values near 0 indicate minimal repayment; values near 1 indicate full repayment. Defaulters are expected to systematically underpay.

In [ ]:
# --- Figure 6: Credit utilisation analysis ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Compute utilisation ratios
util_cols = []
for i in range(1, 7):
    col = f"_UTIL_{i}"
    df[col] = df[f"BILL_AMT{i}"] / df["LIMIT_BAL"].replace(0, np.nan)
    util_cols.append(col)

# (a) September utilisation distribution
ax = axes[0]
for target_val, label, color in [(0, "No Default", C_ND), (1, "Default", C_D)]:
    subset = df[df[TARGET] == target_val]["_UTIL_1"].clip(-0.5, 2)
    ax.hist(subset, bins=80, density=True, alpha=0.35, color=color, label=label)
    subset.plot.kde(ax=ax, color=color, linewidth=2)

ax.axvline(x=1.0, color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.text(1.02, ax.get_ylim()[1] * 0.8, "100% utilisation", fontsize=7, rotation=90, color="gray")
ax.set_xlabel("Credit Utilisation Ratio (BILL_AMT1 / LIMIT_BAL)")
ax.set_ylabel("Density")
ax.set_title("(a) September Utilisation Distribution")
ax.legend()

# (b) Temporal utilisation trajectory
ax = axes[1]
for target_val, label, color in [(0, "No Default", C_ND), (1, "Default", C_D)]:
    subset = df[df[TARGET] == target_val][util_cols]
    means = subset.mean()
    sems = subset.std() / np.sqrt(len(subset))
    ax.plot(MONTH_LABELS, means.values, marker="o", color=color,
            label=label, linewidth=2, markersize=5)
    ax.fill_between(MONTH_LABELS, means.values - 1.96 * sems.values,
                    means.values + 1.96 * sems.values, alpha=0.15, color=color)

ax.set_xlabel("Month (2005)")
ax.set_ylabel("Mean Utilisation Ratio")
ax.set_title("(b) Utilisation Trajectory (95% CI)")
ax.legend()
ax.invert_xaxis()

df.drop(columns=util_cols, inplace=True)

fig.suptitle("Credit Utilisation Analysis -- A Key Risk Indicator",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# --- Figure 13: Repayment ratio ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

repay_cols = []
for i in range(1, 7):
    col = f"_REPAY_{i}"
    denom = df[f"BILL_AMT{i}"].replace(0, np.nan).abs()
    df[col] = (df[f"PAY_AMT{i}"] / denom).clip(0, 3)
    repay_cols.append(col)

# (a) Distribution
ax = axes[0]
for target_val, label, color in [(0, "No Default", C_ND), (1, "Default", C_D)]:
    vals = df[df[TARGET] == target_val]["_REPAY_1"].dropna().clip(0, 2)
    ax.hist(vals, bins=60, density=True, alpha=0.35, color=color, label=label)

ax.axvline(x=1.0, color="gray", linestyle="--", linewidth=1)
ax.text(1.02, ax.get_ylim()[1] * 0.8, "Paid in full", fontsize=7, color="gray", rotation=90)
ax.set_xlabel("Repayment Ratio (PAY_AMT1 / |BILL_AMT1|)")
ax.set_ylabel("Density")
ax.set_title("(a) September Repayment Ratio")
ax.legend()

# (b) Temporal trajectory
ax = axes[1]
for target_val, label, color in [(0, "No Default", C_ND), (1, "Default", C_D)]:
    subset = df[df[TARGET] == target_val][repay_cols]
    medians = subset.median()
    ax.plot(MONTH_LABELS, medians.values, marker="o", color=color,
            label=label, linewidth=2, markersize=5)

ax.set_xlabel("Month (2005)")
ax.set_ylabel("Median Repayment Ratio")
ax.set_title("(b) Repayment Ratio Trajectory")
ax.legend()
ax.invert_xaxis()

df.drop(columns=repay_cols, inplace=True)

fig.suptitle("Repayment Ratio Analysis -- Defaulters Consistently Underpay",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

<a id="9-sequential-dependency-analysis"></a>
## 9. Sequential Dependency Analysis

### 9.1 Feature Interactions

Self-attention computes all $O(n^2)$ pairwise interactions between tokens. We visualise three key interactions that a flat model would miss but attention can capture naturally.

### 9.2 PAY Transition Probabilities

If consecutive PAY values exhibit Markov-like transition dynamics that differ between defaulters and non-defaulters, this provides strong evidence that **positional attention** (knowing which month is which) adds value beyond treating PAY features as an unordered set.

In [ ]:
# --- Figure 10: Feature interactions ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) LIMIT_BAL vs BILL_AMT1 (credit utilisation pattern)
ax = axes[0]
sample = df.sample(n=5000, random_state=42)
ax.scatter(sample[sample[TARGET] == 0]["LIMIT_BAL"],
           sample[sample[TARGET] == 0]["BILL_AMT1"],
           alpha=0.15, s=5, color=C_ND, label="No Default")
ax.scatter(sample[sample[TARGET] == 1]["LIMIT_BAL"],
           sample[sample[TARGET] == 1]["BILL_AMT1"],
           alpha=0.3, s=5, color=C_D, label="Default")
ax.plot([0, 1e6], [0, 1e6], "k--", alpha=0.3, linewidth=1)
ax.text(600000, 650000, "100% utilisation", fontsize=7, color="gray", rotation=35)
ax.set_xlabel("Credit Limit (NT$)")
ax.set_ylabel("September Bill Amount (NT$)")
ax.set_title("(a) Credit Utilisation Pattern")
ax.legend(markerscale=3)

# (b) PAY_0 vs LIMIT_BAL (boxplot)
ax = axes[1]
pay0_vals = sorted(df["PAY_0"].unique())
data_boxes = [df[df["PAY_0"] == v]["LIMIT_BAL"] for v in pay0_vals]
bp = ax.boxplot(data_boxes, positions=range(len(pay0_vals)), widths=0.6,
                patch_artist=True, showfliers=False)
for i, patch in enumerate(bp["boxes"]):
    default_rate = df[df["PAY_0"] == pay0_vals[i]][TARGET].mean()
    patch.set_facecolor(plt.cm.RdYlBu_r(default_rate))
    patch.set_alpha(0.7)
ax.set_xticks(range(len(pay0_vals)))
ax.set_xticklabels(pay0_vals)
ax.set_xlabel("PAY_0 Status")
ax.set_ylabel("Credit Limit (NT$)")
ax.set_title("(b) Credit Limit by Payment Status\n(colour = default rate)")

# (c) Age vs default rate by education
ax = axes[2]
edu_labels = {1: "Grad School", 2: "University", 3: "High School", 4: "Others"}
for edu_val, edu_label in edu_labels.items():
    subset = df[df["EDUCATION"] == edu_val]
    age_bins = pd.cut(subset["AGE"], bins=np.arange(20, 80, 5))
    default_by_age = subset.groupby(age_bins, observed=True)[TARGET].mean()
    midpoints = [interval.mid for interval in default_by_age.index]
    ax.plot(midpoints, default_by_age.values, marker="o", label=edu_label,
            linewidth=1.5, markersize=4)

ax.set_xlabel("Age")
ax.set_ylabel("Default Rate")
ax.set_title("(c) Default Rate by Age x Education")
ax.legend(fontsize=7)

fig.suptitle("Feature Interactions -- Pairwise Patterns Self-Attention Can Capture",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 11: PAY transition probabilities ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for target_val, label, ax in [(0, "No Default", axes[0]), (1, "Default", axes[1])]:
    subset = df[df[TARGET] == target_val]
    transitions = pd.crosstab(
        subset["PAY_2"].clip(-2, 4),
        subset["PAY_0"].clip(-2, 4),
        normalize="index",
    )
    sns.heatmap(transitions, ax=ax, cmap="YlOrRd", annot=True, fmt=".2f",
                linewidths=0.5, annot_kws={"size": 7}, vmin=0, vmax=0.8)
    ax.set_xlabel("PAY_0 (September)")
    ax.set_ylabel("PAY_2 (August)")
    ax.set_title(f"{label}: Aug -> Sep Transition Probabilities")

fig.suptitle("Payment Status Transitions -- Sequential Dependencies\n"
             "Defaulters show higher transition probabilities toward delinquency",
             fontsize=12, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

# --- Quantify transition difference ---
print("\n--- Key Transition Probability Differences ---")
print(f"{'Transition':<25} {'No Default':>12} {'Default':>12} {'Delta':>8}")
print("-" * 60)
for from_val in [-2, -1, 0, 1, 2]:
    for to_val in [0, 1, 2]:
        nd_sub = df[(df[TARGET] == 0) & (df["PAY_2"] == from_val)]
        d_sub = df[(df[TARGET] == 1) & (df["PAY_2"] == from_val)]
        if len(nd_sub) > 50 and len(d_sub) > 50:
            p_nd = (nd_sub["PAY_0"] == to_val).mean()
            p_d = (d_sub["PAY_0"] == to_val).mean()
            if abs(p_d - p_nd) > 0.05:
                print(f"  PAY_2={from_val} -> PAY_0={to_val}  {p_nd:>12.3f} {p_d:>12.3f} {p_d - p_nd:>+8.3f}")

<a id="10-summary-of-findings"></a>
## 10. Summary of Findings and Architectural Implications

| Finding | Evidence | Architectural Implication |
|:---|:---|:---|
| **3.5:1 class imbalance** | 22.1% default rate | Class-weighted BCE or focal loss; evaluate on AUC-ROC, F1, PR-AUC |
| **PAY features dominate** | PAY_0 $|r_{pb}|$ = 0.33, top 6 features are all PAY | Tokenisation must encode PAY status with high fidelity |
| **PAY dual-zone semantics** | Categorical zone $\{-2,-1,0\}$ vs ordinal $\{1,...,8\}$; non-linear risk jump | Hybrid embedding: categorical lookup + ordinal encoding |
| **Temporal divergence** | Significant Spearman trend (p < 0.05) across all 3 feature groups | Sequence model with positional encoding, not bag-of-features |
| **BILL_AMT high autocorrelation** | Adjacent-month $r > 0.95$ | Attention should focus on temporal *changes*, not absolute values |
| **Autocorrelation differs by class** | Decay rate diverges at lag 3+ | Attention heads can learn class-conditional temporal patterns |
| **Credit utilisation is risk-informative** | KS test significant; defaulters have higher UTIL | Include as engineered feature for RF; additional token for transformer |
| **Repayment ratio separates classes** | Defaulters consistently underpay across all 6 months | Strong engineered feature for RF baseline |
| **Transition dynamics differ** | Defaulters: higher P(escalation); lower P(recovery) | Positional attention can learn Markov-like transition structure |
| **Weak demographic signal** | AGE: $|r_{pb}|$ = 0.01; SEX: V = 0.03 | Include but expect attention to down-weight these tokens |

### Conclusion

The data exhibits clear **temporal sequential structure** with class-conditional dynamics. This provides strong empirical justification for applying a transformer architecture --- specifically, self-attention over a sequence of monthly feature tokens with learned positional encoding --- rather than treating the 23 features as an unordered set.

---
*End of Notebook 01. Proceed to Notebook 02 (Data Preprocessing) for the full data preparation pipeline.*